In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================
from urllib.parse import quote

storage_account_name = "cockroachcdc1768934658"  # ← Replace
storage_account_key = "***REMOVED***"      # ← Replace
storage_account_key_encoded = quote(storage_account_key, safe='')  # URL encode for use in connection strings
container_name = "changefeed-events"                 # ← Replace
target_catalog = "main"                              # ← Replace
target_schema = "robert_lee_crdb"                    # ← Replace


In [ ]:
# Configure Azure storage access
# does not work for serverless
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net",
    storage_account_key  # Note: spark.conf.set() doesn't require URL encoding
)


In [ ]:
# ============================================================================
# Create append-only CDC events table using Spark Structured Streaming
# Note: For DLT pipeline version, see stream-changefeed-to-databricks-azure.md
# ============================================================================
from pyspark.sql import functions as F

source_path = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/parquet/defaultdb/public/usertable/"
checkpoint_path = "/checkpoints/usertable/append_only"
target_table = f"{target_catalog}.{target_schema}.usertable_cdc_events"

# Read streaming data with Autoloader
df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
    .option("recursiveFileLookup", "true")
    .load(source_path)
    .select(
        "*",
        F.when(F.col("__crdb__event_type") == "d", "DELETE")
         .otherwise("UPSERT")
         .alias("_cdc_operation"),
        F.col("__crdb__updated").alias("_cdc_timestamp")
    )
)

# Write all CDC events (no deduplication)
query = (df.writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_path}/data")
    .trigger(availableNow=True)  # Process all available data then stop
    .toTable(target_table)
)

print(f"✅ Streaming query started")
print(f"⏳ Processing data...")

query.awaitTermination()

print(f"\n✅ Completed! Query your data:")
print(f"   SELECT * FROM {target_table}")

,message
0,"This Delta Live Tables query is syntactically valid, but you must create a pipeline in order to define and populate your table."
